# 01 - Coleta de Dados da CVM

Este notebook baixa dados da CVM necessários para investigação:

1. **Informe Diário**: PL, cota, captação/resgate diário
2. **CDA**: Composição de carteira mensal
3. **Cadastro**: Dados cadastrais dos fundos

In [1]:
import sys
sys.path.append('..')

from src.collectors.cvm_collector import CVMCollector
from config.settings import Config

## Configuração

In [ ]:
config = Config()
collector = CVMCollector(config)

# Usar período com dados disponíveis (configurado em settings.py)
START_YEAR = config.DEFAULT_START_YEAR
START_MONTH = config.DEFAULT_START_MONTH
END_YEAR = config.DEFAULT_END_YEAR
END_MONTH = config.DEFAULT_END_MONTH

print(f"📅 Período de análise: {START_YEAR}-{START_MONTH:02d} a {END_YEAR}-{END_MONTH:02d}")
print(f"🔍 Verificação de disponibilidade: {'Ativada' if config.DOWNLOAD_CHECK_AVAILABILITY else 'Desativada'}")
print(f"🔄 Máximo de tentativas: {config.DOWNLOAD_MAX_RETRIES}")

# Verificar disponibilidade antes de baixar
print("\n🔎 Verificando disponibilidade de dados na CVM...")
print("(Isto pode levar alguns minutos...)")

available_informe = collector.get_available_months('informe_diario', START_YEAR, END_YEAR)
available_cda = collector.get_available_months('cda', START_YEAR, END_YEAR)

print(f"\n✅ Informe Diário: {len(available_informe)} meses disponíveis")
if available_informe:
    print(f"   Primeiro: {available_informe[0][0]}-{available_informe[0][1]:02d}")
    print(f"   Último: {available_informe[-1][0]}-{available_informe[-1][1]:02d}")

print(f"\n✅ CDA: {len(available_cda)} meses disponíveis")
if available_cda:
    print(f"   Primeiro: {available_cda[0][0]}-{available_cda[0][1]:02d}")
    print(f"   Último: {available_cda[-1][0]}-{available_cda[-1][1]:02d}")

## Download de Informe Diário

In [ ]:
print("📥 Baixando Informe Diário...\n")
print(f"   Período: {START_YEAR}-{START_MONTH:02d} a {END_YEAR}-{END_MONTH:02d}")
print(f"   Verificação de disponibilidade: Ativada\n")

results = collector.download_period(
    start_year=START_YEAR,
    start_month=START_MONTH,
    end_year=END_YEAR,
    end_month=END_MONTH,
    data_types=['informe_diario'],
    check_availability=config.DOWNLOAD_CHECK_AVAILABILITY
)

print(f"\n✅ Total de arquivos baixados: {len(results)}")
if results:
    print("\n📋 Detalhes:")
    for data_type, year, month, path in results:
        if year is not None:
            print(f"   {data_type}: {year}-{month:02d} -> {path.name}")

## Download de CDA (Composição de Carteira)

In [ ]:
print("📥 Baixando CDA (Composição de Carteira)...\n")
print(f"   Período: {START_YEAR}-{START_MONTH:02d} a {END_YEAR}-{END_MONTH:02d}")
print(f"   Nota: CDA disponível apenas a partir de 2023-01\n")

results_cda = collector.download_period(
    start_year=START_YEAR,
    start_month=START_MONTH,
    end_year=END_YEAR,
    end_month=END_MONTH,
    data_types=['cda'],
    check_availability=config.DOWNLOAD_CHECK_AVAILABILITY
)

print(f"\n✅ Total de arquivos baixados: {len(results_cda)}")
if results_cda:
    print("\n📋 Detalhes:")
    for data_type, year, month, path in results_cda:
        if year is not None:
            print(f"   {data_type}: {year}-{month:02d} -> {path.name}")

## Download de Cadastro

In [ ]:
print("📥 Baixando Cadastro...\n")
print("   Nota: Cadastro é um arquivo ÚNICO (não mensal)")
print("   Arquivo: cad_fi.csv (atualizado regularmente pela CVM)\n")

results_cadastro = collector.download_period(
    start_year=START_YEAR,
    start_month=START_MONTH,
    end_year=END_YEAR,
    end_month=END_MONTH,
    data_types=['cadastro'],
    check_availability=config.DOWNLOAD_CHECK_AVAILABILITY
)

print(f"\n✅ Cadastro baixado: {len(results_cadastro) > 0}")
if results_cadastro:
    for data_type, year, month, path in results_cadastro:
        print(f"   Arquivo: {path.name}")
        print(f"   Tamanho: {path.stat().st_size / 1024 / 1024:.2f} MB")

## Verificar Estrutura dos Dados

In [ ]:
import pandas as pd

# Ler uma amostra do Informe Diário
sample_file = config.RAW_DATA_DIR / f"inf_diario_fi_{START_YEAR}{START_MONTH:02d}.csv"

if sample_file.exists():
    df_sample = pd.read_csv(sample_file, encoding='latin1', sep=';', nrows=1000)
    print("\n📊 Colunas disponíveis no Informe Diário:")
    print(df_sample.columns.tolist())
    print(f"\n📏 Shape da amostra: {df_sample.shape}")
    print(f"   {df_sample.shape[0]:,} linhas x {df_sample.shape[1]} colunas")
    print("\n🔍 Primeiras 3 linhas:")
    display(df_sample.head(3))
else:
    print(f"⚠️ Arquivo de amostra não encontrado: {sample_file}")
    print(f"   Arquivos disponíveis em {config.RAW_DATA_DIR}:")
    for f in sorted(config.RAW_DATA_DIR.glob("inf_diario_*.csv")):
        print(f"   - {f.name}")

## Resumo

In [ ]:
print("\n" + "="*60)
print("📊 RESUMO DA COLETA DE DADOS")
print("="*60)

# Contar arquivos baixados
informe_files = len(results) if 'results' in dir() else 0
cda_files = len(results_cda) if 'results_cda' in dir() else 0
cadastro_files = len(results_cadastro) if 'results_cadastro' in dir() else 0

print(f"\n📁 Informe Diário: {informe_files} arquivos")
print(f"📁 CDA: {cda_files} arquivos")
print(f"📁 Cadastro: {cadastro_files} arquivo(s)")

total = informe_files + cda_files + cadastro_files
print(f"\n📊 Total: {total} arquivo(s) baixado(s)")

# Tamanho total
import os
total_size = 0
for file in config.RAW_DATA_DIR.glob("*.csv"):
    total_size += file.stat().st_size

print(f"💾 Tamanho total: {total_size / 1024 / 1024:.2f} MB")
print(f"\n📂 Diretório de dados: {config.RAW_DATA_DIR}")

# Verificar próximos passos
if total > 0:
    print("\n✅ Coleta concluída com sucesso!")
    print("\n📋 Próximos passos:")
    print("   1. Execute: 02_identify_reag_funds.ipynb")
    print("   2. Identifique fundos REAG no cadastro")
    print("   3. Analise fluxos e detecte anomalias")
else:
    print("\n⚠️ Nenhum arquivo foi baixado!")
    print("\n🔧 Possíveis causas:")
    print("   - Problemas de conectividade")
    print("   - Período solicitado não disponível")
    print("   - Verifique os logs acima para detalhes")